# Session 6: Deploying MCP Servers

**Learning goals:**

- Discover tools from two MCP servers and route each call to the client that owns it.
- Deploy a FastMCP server and verify its Streamable HTTP endpoint with an MCP client.
- Deploy a FastAPI web chat that calls the remote MCP server and keeps separate browser histories.

**Prerequisites:**

- [Session 5: Building MCP Server & Client](<../05-Building-MCP-Server-and-Client/Session 5 Building MCP Server & Client.ipynb>) and familiarity with Python, a terminal and virtual environments.
- A local clone of the course repository and a Jupyter kernel in the same environment as the practice dependencies.
- For the interactive chatbot and web chat, a Gemini or OpenRouter API key and a model that supports tool calling. For deployment, a Render account and a Git repository that Render can access.

Use the files in this session folder to build a briefing assistant for a Perth riverside photography walk. The weather server uses the same simulated records as [Adding Prompt and Resource Features in Session 5](<../05-Building-MCP-Server-and-Client/Session 5 Building MCP Server & Client.ipynb#adding-prompt-and-resource-features>). The notebook exercises use those samples and run without API keys.


## Outline

1. [Connect Multiple MCP Servers](#connect-multiple-mcp-servers)
2. [Deploy a Remote MCP Server](#deploy-a-remote-mcp-server)
3. [Deploy the Web Chat](#deploy-the-web-chat)

From the `06-Deploying-MCP-Servers` directory, install [the practice dependencies](practice/requirements.txt), then open this notebook with the same Python environment.

```bash
python -m pip install -r practice/requirements.txt
```


## 1. Connect Multiple MCP Servers

The weather server provides `get_weather(city)`. The [Filesystem server](practice/filesystem_mcp_server.py) provides `read_event_plan()`, which reads the bundled [event plan](sample_data/event-plan.md). The Filesystem server chooses the file, so its callers do not supply a path.

### Connect and inspect each server

Store the server entries in [mcp_servers.json](practice/mcp_servers.json). Each name under `mcpServers` identifies a server. The app uses it as a prefix for tools shown to the model.

```json
{
  "mcpServers": {
    "weather": {
      "args": ["sample_weather_mcp_server.py", "--transport", "stdio"],
      "url_env": "MCP_SERVER_URL",
      "allowed_tools": ["get_weather"]
    },
    "files": {
      "args": ["filesystem_mcp_server.py"],
      "allowed_tools": ["read_event_plan"]
    }
  }
}
```

- `args` gives the script filename and arguments. The app starts local servers with the current Python interpreter and connects over stdio.
- `url_env` names an optional environment variable containing a Streamable HTTP URL. When `MCP_SERVER_URL` is set, the weather client connects to that endpoint. The Filesystem client still uses stdio.
- `allowed_tools` lists the tools the chatbot may pass to the model.

[multi_server_chatbot.py](practice/multi_server_chatbot.py) reads the JSON and creates one client per entry. `connect_to_servers()` loops over those clients and calls [Session 5's `connect_to_server()`](../05-Building-MCP-Server-and-Client/practice/mcp_chatbot.py) for each. The single-server method opens one connection and discovers its tools. An `AsyncExitStack` keeps all connections open until the caller finishes and closes any that already opened if a later connection fails.

Run the next cell to inspect the configured servers and call one tool from each. Leave `MCP_SERVER_URL` unset for the local stdio exercise. Tool calls in this cell need no LLM API key.


In [ ]:
import sys

# Make the session’s practice modules importable from the notebook.
sys.path.insert(0, "practice")
from multi_server_chatbot import MultiServerChatbot

chatbot = MultiServerChatbot()
print("Configured servers:", list(chatbot.servers))

async with chatbot.connect_to_servers():
    print("Routed tools:", list(chatbot.routes))

    # Use the prefixed names the model sees to call the owning server.
    print(await chatbot.execute_tool("weather_get_weather", {"city": "Perth"}))
    print(await chatbot.execute_tool("files_read_event_plan", {}))


### Extend the Session 5 chatbot

The Session 6 chatbot inherits the Session 5 chat loop for model requests, streamed replies and conversation history. Once each client is open, `connect_to_servers()` checks its discovered tool names against the JSON allowlist. It prefixes each allowed tool's name and records the owning client and original name.

| Name sent to the model | MCP client | Original tool name |
|---|---|---|
| `weather_get_weather` | Weather | `get_weather` |
| `files_read_event_plan` | Filesystem | `read_event_plan` |

When the model requests a tool, `execute_tool()` looks up the prefixed name and sends the original name and arguments to the owning client. The inherited chat loop passes the result back under the model's original tool-call ID. The prefix keeps tools with the same original name distinct across servers.


### Try the combined workflow

Set these environment variables in the terminal that will run the chatbot. Choose one provider and set its matching key. Use a model ID from that provider's catalogue that supports tool calling.

| Variable | Value |
|---|---|
| `LLM_PROVIDER` | `openrouter` or `gemini` |
| `CHAT_MODEL` | A model ID that supports tool calling with your provider |
| `OPENROUTER_API_KEY` or `GEMINI_API_KEY` | Your key for the chosen provider |

Keep keys in the terminal or hosting environment. From the session directory, start the chatbot.

```bash
python practice/multi_server_chatbot.py
```

Ask **Brief me on the Perth photography walk. Use the event plan and simulated weather.** Check that the terminal logs show calls to both prefixed tools. The briefing should include Elizabeth Quay, 9 am, and the simulated weather of 24 °C and partly cloudy conditions. Ask where the group should meet if it rains, then type `quit` to close both clients.


## 2. Deploy a Remote MCP Server

The [weather MCP server](practice/sample_weather_mcp_server.py) exposes simulated weather over Streamable HTTP. Test it on your computer first, then deploy the same server to Render. It needs no API keys.

### Test the MCP server locally

From the session directory, start the server in a terminal.

```bash
python practice/sample_weather_mcp_server.py --host 127.0.0.1 --port 8000
```

Open `http://127.0.0.1:8000/health` and check for `OK`. Then set `CHECK_HTTP` to `True` and run the cell below. The MCP client should find `get_weather` and read the Perth sample. Leave the local server running for section 3.


In [ ]:
from fastmcp import Client

CHECK_HTTP = False  # Enable after starting the weather server.
mcp_url = "http://127.0.0.1:8000/mcp"

if CHECK_HTTP:
    async with Client(mcp_url, timeout=30) as weather:
        print("Tools:", [tool.name for tool in await weather.list_tools()])
        sample, = await weather.read_resource("weather://sample/perth")
        print(sample.text)


### Create the Render weather service

1. Push the course exercise files to a branch in your own repository that Render can access.
2. In the Render Dashboard, choose **New → Web Service**. Select your repository and the branch containing the exercise files.
3. Enter the settings below and deploy. You can set the health check path under the service's **Settings → Health Checks**.

| Setting | Value |
|---|---|
| Name | A unique name for your weather service |
| Language | Python 3 |
| Root Directory | `06-Deploying-MCP-Servers` |
| Build Command | `python -m pip install -r practice/requirements.txt` |
| Start Command | `python practice/sample_weather_mcp_server.py --host 0.0.0.0 --port "$PORT"` |
| Health Check Path | `/health` |

Render runs the commands from the selected root directory. The server listens on `0.0.0.0` and uses the `PORT` that Render provides.

[Render Web Services](https://render.com/docs/web-services) · [Render health checks](https://render.com/docs/health-checks)

### Verify the deployed MCP endpoint

When the deploy succeeds, copy the weather service's HTTPS URL from its Render dashboard. Open that URL with `/health` appended and check for `OK`. In the test cell above, replace `mcp_url` with the weather service's HTTPS URL ending in `/mcp` and run the cell again. Expect the same tool and Perth sample. Keep the full HTTPS `/mcp` URL for the Render deployment in section 3.


## 3. Deploy the Web Chat

[The FastAPI web chat](practice/web_chat.py) connects to the weather MCP server over Streamable HTTP and runs the Filesystem server as a subprocess. Test the browser, FastAPI and both MCP servers locally before deploying the web chat as a second Render service.

| Connection | Interface |
|---|---|
| Browser → web chat service | Same-origin `POST /chat` |
| Web chat service → LLM provider | HTTPS chat completions |
| Web chat service → weather service | Streamable HTTP at the weather server's `/mcp` URL |
| Web chat service → Filesystem server | stdio to the bundled subprocess |

### Test the web chat locally

Keep the local weather server from section 2 running. In a new terminal, set the provider variables from section 1 and set `MCP_SERVER_URL` to `http://127.0.0.1:8000/mcp`. From the session directory, start FastAPI.

```bash
python -m uvicorn web_chat:app --app-dir practice --host 127.0.0.1 --port 8080
```

Open [the local web chat](http://127.0.0.1:8080/) and ask **Brief me on the Perth photography walk. Use the event plan and simulated weather.** Check for the Elizabeth Quay meeting point and the simulated Perth weather. Ask **What should I bring?** to check that the browser can continue the conversation.

### Create the Render web chat service

In the Render Dashboard, choose **New → Web Service** again. Select the same repository and branch. Leave **Root Directory** blank so the service can load both the Session 6 app and the Session 5 chatbot module it imports.

| Setting | Value |
|---|---|
| Name | A unique name for your web chat service |
| Language | Python 3 |
| Root Directory | Leave blank |
| Build Command | `python -m pip install -r 06-Deploying-MCP-Servers/practice/requirements.txt` |
| Start Command | `python -m uvicorn web_chat:app --app-dir 06-Deploying-MCP-Servers/practice --host 0.0.0.0 --port "$PORT" --workers 1` |
| Health Check Path | `/health` |

[Deploy a FastAPI app on Render](https://render.com/docs/deploy-fastapi) · [Render root directory settings](https://render.com/docs/monorepo-support)


### Connect the two Render services

Before deploying the web chat, add the following variables in its **Environment** settings. If the service already exists, save the variables and deploy again.

| Variable | Value |
|---|---|
| `MCP_SERVER_URL` | The weather service's full HTTPS URL ending in `/mcp` |
| `LLM_PROVIDER` | `openrouter` or `gemini` |
| `CHAT_MODEL` | A model ID from the chosen provider that supports tool calling |
| `OPENROUTER_API_KEY` or `GEMINI_API_KEY` | The key for the chosen provider |
| `CHAT_ACCESS_PASSWORD` | A strong classroom password using ASCII characters |

The web chat uses its own HTTPS URL. Its provider key stays in the Render service environment.

[Render environment variables](https://render.com/docs/configure-environment-variables)

### Verify the deployed web chat

1. Open the web chat service's HTTPS URL and sign in as `learner` using the classroom password.
2. Ask **Brief me on the Perth photography walk. Use the event plan and simulated weather.** Check for the Elizabeth Quay meeting point and weather labelled as simulated.
3. Ask **What should I bring?** in the same browser. Open a private window and sign in to check that it starts with a separate conversation.
4. If the chat fails, inspect the web service's Render logs and verify its model settings and the weather service's `/mcp` URL.

The single worker keeps browser histories in one process for the classroom demo. Histories reset when the web chat service restarts. Stop the two Render services when you finish the exercise.
